# 注意力机制

本 notebook 通过最小实现验证 Scaled Dot-Product Attention 和 Multi-Head Attention 的计算逻辑，并可视化注意力权重分布。

对应文档：[README.md](README.md)

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import matplotlib

# 环境检查
torch.manual_seed(42)
print(f"PyTorch 版本: {torch.__version__}")
print(f"GPU 可用: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前设备: {device}")

## 1. 直觉

想象你在一堆文档里检索信息：
- **Query**：你的检索请求（"猫在哪里"）
- **Key**：每份文档的索引标签（"sat on the mat", "looked at the bird"...）
- **Value**：文档的真实内容

注意力机制计算 Query 和每个 Key 的相关性，然后按相关性加权汇聚 Value。

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

> 你要记住：`QK^T` 给相关性，`softmax` 给权重，`@V` 给上下文聚合结果。

In [ ]:
# 核心思想：实现缩放点积注意力的最小版本
# 数学对应：Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) @ V
# 复杂度：时间 O(n²d)，空间 O(n²)

def scaled_dot_product_attention(q, k, v, mask=None):
    """
    Args:
        q: (batch, heads, seq_q, d_k)
        k: (batch, heads, seq_k, d_k)
        v: (batch, heads, seq_k, d_v)
        mask: (batch, 1, 1, seq_k) 或 None
    Returns:
        context: (batch, heads, seq_q, d_v)
        weights: (batch, heads, seq_q, seq_k)
    """
    d_k = q.size(-1)

    # Step 1: 计算相关性得分，除以 sqrt(d_k) 防止 softmax 过尖锐
    scores = (q @ k.transpose(-2, -1)) / math.sqrt(d_k)  # (batch, heads, seq_q, seq_k)

    # Step 2: 应用掩码（将被遮蔽位置设为 -inf，softmax 后趋近于 0）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 3: Softmax 归一化，得到注意力权重
    weights = torch.softmax(scores, dim=-1)

    # Step 4: 加权聚合 Value
    context = weights @ v  # (batch, heads, seq_q, d_v)

    return context, weights

## 2. 验证：单头注意力

In [ ]:
# 构造一个简单的单头注意力场景
BATCH, HEADS, SEQ_LEN, D_K = 1, 1, 6, 64

q = torch.randn(BATCH, HEADS, SEQ_LEN, D_K)
k = torch.randn(BATCH, HEADS, SEQ_LEN, D_K)
v = torch.randn(BATCH, HEADS, SEQ_LEN, D_K)

context, weights = scaled_dot_product_attention(q, k, v)

print(f"输入 Q shape:      {q.shape}")
print(f"注意力权重 shape:  {weights.shape}")
print(f"输出 context shape:{context.shape}")
print(f"权重行和（应为 1.0）: {weights.sum(dim=-1)}")

## 3. 多头注意力实现

In [ ]:
# 核心思想：将隐藏维分成多个子空间并行计算注意力，捕捉不同类型的关系
# 数学对应：MultiHead(Q,K,V) = Concat(head_1,...,head_h) W_O，其中 head_i = Attention(QW_i^Q, KW_i^K, VW_i^V)
# 复杂度：时间 O(n²d)，空间 O(n²)，与单头相同（头数不改变总计算量）

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        """
        Args:
            d_model: 隐藏维度，必须能被 num_heads 整除
            num_heads: 头数
        """
        super().__init__()
        assert d_model % num_heads == 0, f"d_model ({d_model}) 必须能被 num_heads ({num_heads}) 整除"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度

        # 四个线性投影：Q、K、V 的输入投影 + 输出投影
        self.w_q = nn.Linear(d_model, d_model, bias=False)
        self.w_k = nn.Linear(d_model, d_model, bias=False)
        self.w_v = nn.Linear(d_model, d_model, bias=False)
        self.w_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, query, key, value, mask=None):
        """
        Args:
            query: (batch, seq_q, d_model)
            key:   (batch, seq_k, d_model)
            value: (batch, seq_k, d_model)
            mask:  (batch, 1, 1, seq_k) 或 None
        Returns:
            out:     (batch, seq_q, d_model)
            weights: (batch, num_heads, seq_q, seq_k)
        """
        bsz = query.size(0)

        # 线性投影后 reshape 成多头形式：(batch, seq, d_model) → (batch, heads, seq, d_k)
        q = self.w_q(query).view(bsz, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.w_k(key).view(bsz, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.w_v(value).view(bsz, -1, self.num_heads, self.d_k).transpose(1, 2)

        # 各头独立计算注意力
        context, weights = scaled_dot_product_attention(q, k, v, mask)

        # 拼接各头输出，再经输出投影：(batch, heads, seq, d_k) → (batch, seq, d_model)
        context = context.transpose(1, 2).contiguous().view(bsz, -1, self.d_model)
        out = self.w_o(context)

        return out, weights

In [ ]:
# 验证多头注意力
D_MODEL, NUM_HEADS, SEQ_LEN = 512, 8, 10

mha = MultiHeadAttention(d_model=D_MODEL, num_heads=NUM_HEADS)
x = torch.randn(1, SEQ_LEN, D_MODEL)  # batch=1

out, weights = mha(x, x, x)  # 自注意力：Q=K=V=x

print(f"输入 shape:        {x.shape}")
print(f"输出 shape:        {out.shape}  （应与输入相同）")
print(f"注意力权重 shape:  {weights.shape}  （batch, heads, seq_q, seq_k）")
print(f"参数量:            {sum(p.numel() for p in mha.parameters()):,}")

## 4. 进一步探索：可视化注意力权重

In [ ]:
# 构造一个有语义的简单场景：句子中各 token 的注意力热力图
tokens = ["The", "cat", "sat", "on", "the", "mat"]
SEQ = len(tokens)

mha_vis = MultiHeadAttention(d_model=64, num_heads=4)
x_vis = torch.randn(1, SEQ, 64)
_, attn_weights = mha_vis(x_vis, x_vis, x_vis)

# 取第 0 个 batch，展示所有 4 个头
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("各注意力头的权重热力图（随机初始化，仅展示可视化方式）", fontsize=13)

for h, ax in enumerate(axes):
    w = attn_weights[0, h].detach().numpy()  # (seq, seq)
    im = ax.imshow(w, vmin=0, vmax=w.max(), cmap="Blues")
    ax.set_xticks(range(SEQ))
    ax.set_yticks(range(SEQ))
    ax.set_xticklabels(tokens, rotation=45, ha="right")
    ax.set_yticklabels(tokens)
    ax.set_title(f"Head {h+1}")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
print("提示：实际训练后的模型中，不同头会关注不同的语言模式（语法/语义/位置等）")

## 5. Causal Mask 验证

自回归生成任务中，token 只能看到自身及之前的 token，不能看到未来。

In [ ]:
# 核心思想：构造下三角掩码，强制因果约束
# 数学对应：mask[i,j] = 1 if j <= i else 0，即位置 i 只能关注位置 0..i
# 复杂度：掩码构造 O(n²)，不增加注意力计算复杂度

SEQ = 6
causal_mask = torch.tril(torch.ones(1, 1, SEQ, SEQ))  # 下三角矩阵

print(f"Causal Mask (形状 {causal_mask.shape}):")
print(causal_mask[0, 0].int())
print("\n第 i 行表示：位置 i 能关注的位置（1=可见，0=遮蔽）")

# 验证掩码生效
q = torch.randn(1, 1, SEQ, 32)
k = torch.randn(1, 1, SEQ, 32)
v = torch.randn(1, 1, SEQ, 32)

_, weights_causal = scaled_dot_product_attention(q, k, v, mask=causal_mask)
print(f"\n应用 Causal Mask 后，权重上三角应为 0:")
print(weights_causal[0, 0].detach().round(decimals=4))